# Distribution Fitting, Transformations, And Moment Ideas

**Official MA1001B Alignment:** *3.3 normal distribution; 3.3 gamma-type distributions; 3.4 moment-generating function.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Fit parametric statistical distributions (Normal, Gamma) to empirical measurement data using maximum likelihood estimation.
- Apply logarithmic transformations to normalize skewed positive variables for statistical modeling.
- Compare statistical moments (mean, variance, skewness) across original and transformed measurement scales.
- Translate log-scale model inferences back into original operational units for stakeholder decision making.


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We fit parametric probability models to capture essential distributional moments and tail behaviors.
- **2. Computational Link (How Python represents it):** We use SciPy distribution fitting (`stats.norm.fit`, `stats.gamma.fit`) and logarithmic transformations (`np.log`, `np.exp`).
- **3. Decision Link (How it guides action):** Selecting the appropriate parametric model ensures that tail-risk estimates and forecasting intervals are reliable.


## Decision Scenario

> **The Problem:** An analyst must decide whether to model a positive measurement on its original scale or after a log transformation. The choice affects forecasts and intervals.


## Conceptual Explanation

Distribution fitting is not about forcing data to match a named curve. It is about deciding whether a model captures the features that matter for the decision. Transformations can make skewed positive data easier to model, but conclusions must be translated back carefully.


## Mathematical Anchor

Moments summarize distribution shape. The first moment relates to center, the second to spread, and higher moments to skewness or tail behavior.


## Data And Workflow Notes

Uses a skewed positive measurement so students can compare original and log scales.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: Simulating Skewed Data & Log Transformation

We generate a skewed positive dataset and compute its natural logarithm, creating two parallel scales for modeling comparison.


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Simulate skewed positive measurement values and create log-transformed scale
values = pd.Series(rng.lognormal(mean=11.9, sigma=0.45, size=1000), name="original_value")
log_values = np.log(values)

# Display head of both scales
pd.DataFrame({"original_scale": values, "log_scale": log_values}).head()


### Step 2: Comparing Statistical Moments Across Scales

We calculate and compare the first three statistical moments (mean, standard deviation, skewness) on both the original and logarithmic scales.


In [ ]:
# Contrast moments across original and log-transformed scales
comparison = pd.DataFrame({
    "scale": ["original_positive_units", "log_transformed_units"],
    "mean_1st_moment": [values.mean(), log_values.mean()],
    "std_2nd_moment": [values.std(ddof=1), log_values.std(ddof=1)],
    "skewness_3rd_moment": [stats.skew(values), stats.skew(log_values)],
})
comparison.round(3)


### Step 3: Visualizing Scale Transformation Effects

We plot side-by-side histograms with KDE curves to visually verify how the logarithmic transformation removes right-skewness and restores symmetry.


In [ ]:
# Plot side-by-side comparison of original vs log-transformed distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(values, kde=True, ax=axes[0], color="indianred", bins=30)
axes[0].set_title("Original Scale (Right-Skewed)", fontsize=13)
axes[0].set_xlabel("Original Units")

sns.histplot(log_values, kde=True, ax=axes[1], color="teal", bins=30)
axes[1].set_title("Logarithmic Scale (Symmetric / Normal)", fontsize=13)
axes[1].set_xlabel("Log Units")
plt.tight_layout()
plt.show()


### Step 4: Parametric Fitting & Tail Risk Evaluation

We fit a Normal distribution to the log scale and a Gamma distribution to the original scale, then compare how accurately each model predicts top 10% tail risk.


In [ ]:
# Fit Normal model to log scale and Gamma model to original scale
normal_fit_log = stats.norm.fit(log_values)  # returns (loc, scale)
gamma_fit_original = stats.gamma.fit(values, floc=0)  # returns (shape, loc, scale)

# Evaluate tail probability above the empirical 90th percentile
threshold_90 = values.quantile(0.90)
prob_large_original = (values > threshold_90).mean()

# Translate threshold to log scale to check Normal model survival function
normal_model_tail_prob = stats.norm.sf(np.log(threshold_90), loc=normal_fit_log[0], scale=normal_fit_log[1])
gamma_model_tail_prob = stats.gamma.sf(threshold_90, *gamma_fit_original)

pd.Series({
    "empirical_tail_probability": prob_large_original,
    "lognormal_model_tail_prediction": normal_model_tail_prob,
    "gamma_model_tail_prediction": gamma_model_tail_prob,
}).round(4)


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> Why might the log scale be better for statistical modeling but the original scale better for stakeholder communication?

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Choosing a distribution solely because its histogram looks visually similar without checking tail behavior.
- **Warning:** Forgetting to apply the inverse transformation (`np.exp`) when reporting log-scale model conclusions back to stakeholders.
- **Warning:** Treating fitted parametric coefficients as absolute universal truths rather than sample-dependent estimates.


## Independent Practice

> [!TIP]
> **Your Task:**
> Fit an Exponential or Weibull distribution (`stats.expon.fit`, `stats.weibull_min.fit`) to `values`. Compare its 90th percentile tail prediction against the empirical data.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** What essential feature of a probability distribution is measured by the third statistical moment (skewness)?

*Write your brief conceptual reflection below:*
